# Notebook 20 — Policy Engine & Official Game Integration

## The Pokémon Company — PTCG AI Battle Challenge

### Team Jesus

This notebook builds the production decision engine that converts an official
Kaggle `Observation` into the best legal official action index.

## Pipeline

```text
Observation dictionary
        ↓
to_observation_class()
        ↓
BattleSnapshot
        ↓
Battle features
        ↓
Legal-action ranking
        ↓
Policy safety checks
        ↓
Official option index

# Cell 2 — Imports

In [1]:
from __future__ import annotations

import importlib.util
import sys
import types

from dataclasses import dataclass
from pathlib import Path
from typing import Any

import pandas as pd

print("Python:", sys.version)
print("Current directory:", Path.cwd())

Python: 3.13.3 (tags/v3.13.3:6280bb5, Apr  8 2025, 14:47:33) [MSC v.1943 64 bit (AMD64)]
Current directory: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\notebooks


# Cell 3 — Locate the project

In [2]:
def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()

    markers = [
        "src",
        "notebooks",
        "scripts",
        "data",
    ]

    for candidate in [current, *current.parents]:
        marker_count = sum(
            (candidate / marker).exists()
            for marker in markers
        )

        if marker_count >= 3:
            return candidate

    if current.name.lower() == "notebooks":
        return current.parent

    return current


PROJECT_ROOT = find_project_root()

SRC_DIR = PROJECT_ROOT / "src"
SCRIPTS_DIR = PROJECT_ROOT / "scripts"

NOTEBOOK18_EXPORT = (
    SCRIPTS_DIR
    / "18_kaggle_observation_adapter.py"
)

NOTEBOOK19_EXPORT = (
    SCRIPTS_DIR
    / "19_battle_feature_extraction.py"
)

POLICY_ENGINE_DIR = (
    SRC_DIR
    / "policy_engine"
)

NOTEBOOK20_REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "notebook20"
)

for directory in [
    POLICY_ENGINE_DIR,
    NOTEBOOK20_REPORT_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Notebook 18 export:", NOTEBOOK18_EXPORT)
print("Notebook 19 export:", NOTEBOOK19_EXPORT)
print("Policy engine:", POLICY_ENGINE_DIR)
print("Notebook 20 reports:", NOTEBOOK20_REPORT_DIR)

Project root: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge
Notebook 18 export: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\scripts\18_kaggle_observation_adapter.py
Notebook 19 export: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\scripts\19_battle_feature_extraction.py
Policy engine: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\src\policy_engine
Notebook 20 reports: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook20


# Cell 4 — Load Notebook 18

In [4]:
# Cell 4 — Load Notebook 18 safely

import sys
import types


if not NOTEBOOK18_EXPORT.is_file():
    raise FileNotFoundError(
        f"Notebook 18 export not found:\n{NOTEBOOK18_EXPORT}"
    )

module_name_18 = "notebook18_adapter"

source_18 = NOTEBOOK18_EXPORT.read_text(
    encoding="utf-8-sig"
)

source_18_lines = source_18.splitlines()

cleaned_18_lines = [
    line
    for line in source_18_lines
    if line.strip() != "from __future__ import annotations"
]

cleaned_18_source = (
    "from __future__ import annotations\n"
    + "\n".join(cleaned_18_lines)
)

notebook18 = types.ModuleType(module_name_18)
notebook18.__file__ = str(NOTEBOOK18_EXPORT)
notebook18.__package__ = ""

sys.modules[module_name_18] = notebook18

compiled_18 = compile(
    cleaned_18_source,
    str(NOTEBOOK18_EXPORT),
    "exec",
)

exec(
    compiled_18,
    notebook18.__dict__,
)

print("Notebook 18 loaded successfully.")
print(
    "Repeated future imports removed:",
    len(source_18_lines) - len(cleaned_18_lines),
)

Python: 3.13.3 (tags/v3.13.3:6280bb5, Apr  8 2025, 14:47:33) [MSC v.1943 64 bit (AMD64)]
Current directory: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\notebooks
Project root: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge
Card database: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\src\card_database
Observation adapter: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\src\observation_adapter
Notebook 18 reports: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook18
Repository size: 1267
Test card: 678 Mega Lucario ex
Move count: 2

Notebook 17 package integration passed.
Candidate cg directories:
FOUND   D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\submission_work\official_sample_extracted\cg
FOUND   D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\submission_work\team_jesus_baseline\cg
MISSING D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\data\raw\kaggle_sample_submi

,name,module,annotations
18,ApiResult,cg.api,"[state, error]"
1,AreaType,cg.api,[]
21,Attack,cg.api,"[attackId, name, text, damage, energies]"
9,Card,cg.api,"[id, serial, playerIndex]"
20,CardData,cg.api,"[cardId, name, cardType, retreatCost, hp, weak..."
3,CardType,cg.api,[]
2,EnergyType,cg.api,[]
0,IntEnum,enum,[]
15,Log,cg.api,"[type, playerIndex, hasBasicPokemon, cardId, s..."
8,LogType,cg.api,[]


Important official classes:

- Card
- CardData
- CardType
- Log
- LogType
- Option
- OptionType
- PlayerState
- Pokemon
- SearchState
- SelectContext
- SelectData
- SelectType
- State

Card
Class: <class 'cg.api.Card'>

Annotations:
- id: <class 'int'>
- serial: <class 'int'>
- playerIndex: <class 'int'>

CardData
Class: <class 'cg.api.CardData'>

Annotations:
- cardId: <class 'int'>
- name: <class 'str'>
- cardType: <enum 'CardType'>
- retreatCost: <class 'int'>
- hp: <class 'int'>
- weakness: cg.api.EnergyType | None
- resistance: cg.api.EnergyType | None
- energyType: <enum 'EnergyType'>
- basic: <class 'bool'>
- stage1: <class 'bool'>
- stage2: <class 'bool'>
- ex: <class 'bool'>
- megaEx: <class 'bool'>
- tera: <class 'bool'>
- aceSpec: <class 'bool'>
- evolvesFrom: str | None
- skills: list[cg.api.Skill]
- attacks: list[int]

CardType
Class: <enum 'CardType'>

No annotations found.

Public attributes:
- BASIC_ENERGY
- ITEM
- POKEMON
- SPECIAL_ENERGY
- STADIUM
- SUPPORTER
- TOOL
-

,name,value
0,DECK,1
1,HAND,2
2,DISCARD,3
3,ACTIVE,4
4,BENCH,5
5,PRIZE,6
6,STADIUM,7
7,ENERGY,8
8,TOOL,9
9,PRE_EVOLUTION,10


OptionType


,name,value
0,NUMBER,0
1,YES,1
2,NO,2
3,CARD,3
4,TOOL_CARD,4
5,ENERGY_CARD,5
6,ENERGY,6
7,PLAY,7
8,ATTACH,8
9,EVOLVE,9


SelectContext


,name,value
0,MAIN,0
1,SETUP_ACTIVE_POKEMON,1
2,SETUP_BENCH_POKEMON,2
3,SWITCH,3
4,TO_ACTIVE,4
5,TO_BENCH,5
6,TO_FIELD,6
7,TO_HAND,7
8,DISCARD,8
9,TO_DECK,9


Official cg card objects: 1267
Notebook 17 repository: 1267

Official unique Card IDs: 1267
Repository unique Card IDs: 1267
Missing from repository: 0
Extra in repository: 0
Official card sample 1
Type: <class 'cg.api.CardData'>
aceSpec: False
attacks: []
basic: False
cardId: 1
cardType: 5
energyType: 1
evolvesFrom: None
ex: False
hp: 0
megaEx: False
name: Basic {G} Energy
resistance: None
retreatCost: 0
skills: []
stage1: False
stage2: False
tera: False
weakness: None

Official card sample 2
Type: <class 'cg.api.CardData'>
aceSpec: False
attacks: []
basic: False
cardId: 2
cardType: 5
energyType: 2
evolvesFrom: None
ex: False
hp: 0
megaEx: False
name: Basic {R} Energy
resistance: None
retreatCost: 0
skills: []
stage1: False
stage2: False
tera: False
weakness: None

Official card sample 3
Type: <class 'cg.api.CardData'>
aceSpec: False
attacks: []
basic: False
cardId: 3
cardType: 5
energyType: 3
evolvesFrom: None
ex: False
hp: 0
megaEx: False
name: Basic {W} Energy
resistance: None
retr

# Cell 5 — Load Notebook 19

In [8]:
spec19 = importlib.util.spec_from_file_location(
    "notebook19_features",
    NOTEBOOK19_EXPORT,
)

notebook19 = importlib.util.module_from_spec(spec19)
spec19.loader.exec_module(notebook19)

print("Notebook 19 loaded.")

Python: 3.13.3 (tags/v3.13.3:6280bb5, Apr  8 2025, 14:47:33) [MSC v.1943 64 bit (AMD64)]
Current directory: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\notebooks
Project root: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge
Notebook 18 export: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\scripts\18_kaggle_observation_adapter.py
Decision-feature package: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\src\decision_features
Notebook 19 reports: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook19
Repository size: 1267
Test card: 678 Mega Lucario ex
Move count: 2

Card database integration passed.
Python: 3.13.3 (tags/v3.13.3:6280bb5, Apr  8 2025, 14:47:33) [MSC v.1943 64 bit (AMD64)]
Current directory: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\notebooks
Project root: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge
Card database: D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_

,name,module,annotations
18,ApiResult,cg.api,"[state, error]"
1,AreaType,cg.api,[]
21,Attack,cg.api,"[attackId, name, text, damage, energies]"
9,Card,cg.api,"[id, serial, playerIndex]"
20,CardData,cg.api,"[cardId, name, cardType, retreatCost, hp, weak..."
3,CardType,cg.api,[]
2,EnergyType,cg.api,[]
0,IntEnum,enum,[]
15,Log,cg.api,"[type, playerIndex, hasBasicPokemon, cardId, s..."
8,LogType,cg.api,[]


Important official classes:

- Card
- CardData
- CardType
- Log
- LogType
- Option
- OptionType
- PlayerState
- Pokemon
- SearchState
- SelectContext
- SelectData
- SelectType
- State

Card
Class: <class 'cg.api.Card'>

Annotations:
- id: <class 'int'>
- serial: <class 'int'>
- playerIndex: <class 'int'>

CardData
Class: <class 'cg.api.CardData'>

Annotations:
- cardId: <class 'int'>
- name: <class 'str'>
- cardType: <enum 'CardType'>
- retreatCost: <class 'int'>
- hp: <class 'int'>
- weakness: cg.api.EnergyType | None
- resistance: cg.api.EnergyType | None
- energyType: <enum 'EnergyType'>
- basic: <class 'bool'>
- stage1: <class 'bool'>
- stage2: <class 'bool'>
- ex: <class 'bool'>
- megaEx: <class 'bool'>
- tera: <class 'bool'>
- aceSpec: <class 'bool'>
- evolvesFrom: str | None
- skills: list[cg.api.Skill]
- attacks: list[int]

CardType
Class: <enum 'CardType'>

No annotations found.

Public attributes:
- BASIC_ENERGY
- ITEM
- POKEMON
- SPECIAL_ENERGY
- STADIUM
- SUPPORTER
- TOOL
-

,name,value
0,DECK,1
1,HAND,2
2,DISCARD,3
3,ACTIVE,4
4,BENCH,5
5,PRIZE,6
6,STADIUM,7
7,ENERGY,8
8,TOOL,9
9,PRE_EVOLUTION,10


OptionType


...

SelectContext


,name,value
0,MAIN,0
1,SETUP_ACTIVE_POKEMON,1
2,SETUP_BENCH_POKEMON,2
3,SWITCH,3
4,TO_ACTIVE,4
5,TO_BENCH,5
6,TO_FIELD,6
7,TO_HAND,7
8,DISCARD,8
9,TO_DECK,9


Official cg card objects: 1267
Notebook 17 repository: 1267

Official unique Card IDs: 1267
Repository unique Card IDs: 1267
Missing from repository: 0
Extra in repository: 0
Official card sample 1
Type: <class 'cg.api.CardData'>
aceSpec: False
attacks: []
basic: False
cardId: 1
cardType: 5
energyType: 1
evolvesFrom: None
ex: False
hp: 0
megaEx: False
name: Basic {G} Energy
resistance: None
retreatCost: 0
skills: []
stage1: False
stage2: False
tera: False
weakness: None

Official card sample 2
Type: <class 'cg.api.CardData'>
aceSpec: False
attacks: []
basic: False
cardId: 2
cardType: 5
energyType: 2
evolvesFrom: None
ex: False
hp: 0
megaEx: False
name: Basic {R} Energy
resistance: None
retreatCost: 0
skills: []
stage1: False
stage2: False
tera: False
weakness: None

Official card sample 3
Type: <class 'cg.api.CardData'>
aceSpec: False
attacks: []
basic: False
cardId: 3
cardType: 5
energyType: 3
evolvesFrom: None
ex: False
hp: 0
megaEx: False
name: Basic {W} Energy
resistance: None
retr

,rank,option_index,option_type,semantic_label,score,reasons
0,1,0,ATTACK,Attack with Mega Lucario ex,259.913,base=100.0 | attack_bonus=150.0 | active_energ...
1,2,1,END,End Turn,-25.000,base=0.0 | end_turn_penalty=-25.0


[OK] player_feature_extraction
[OK] battle_feature_extraction
[OK] action_feature_extraction
[OK] legal_action_ranking
[OK] best_option_index

Notebook 19 validation passed.
Notebook 19 loaded.


# Cell 6 — Import the production functions

In [9]:
adapt_observation = notebook18.adapt_observation

extract_player_features = notebook19.extract_player_features
extract_battle_features = notebook19.extract_battle_features

rank_legal_actions = notebook19.rank_legal_actions
choose_best_option_index = notebook19.choose_best_option_index

print("Production functions imported.")

Production functions imported.


# Cell 7 — Verify everything

In [10]:
print(callable(adapt_observation))
print(callable(extract_player_features))
print(callable(extract_battle_features))
print(callable(rank_legal_actions))
print(callable(choose_best_option_index))

assert callable(adapt_observation)
assert callable(rank_legal_actions)
assert callable(choose_best_option_index)

print("\nNotebook dependencies verified.")

True
True
True
True
True

Notebook dependencies verified.


# Cell 8 — Build the BattlePolicy class

In [11]:
from dataclasses import dataclass
from typing import Any


@dataclass(slots=True)
class BattlePolicy:
    repository: Any
    official_lookup: dict[int, Any]
    debug: bool = False

    def choose_action(self, observation):
        """
        Convert an official Kaggle Observation into the
        best legal official option index.
        """

        snapshot = adapt_observation(
            observation,
            repository=self.repository,
            official_lookup=self.official_lookup,
        )

        ranked = rank_legal_actions(
            snapshot,
            repository=self.repository,
        )

        if self.debug:

            print("=" * 70)
            print("Ranked legal actions")
            print("=" * 70)

            for action in ranked:
                print(action)

        return choose_best_option_index(
            snapshot,
            repository=self.repository,
        )

# Cell 9A — Retrieve shared dependencies:

In [13]:
from src.card_database import load_repository

repository = load_repository()

official_card_data_by_id = getattr(
    notebook18,
    "official_card_data_by_id",
    None,
)

if official_card_data_by_id is None:
    official_cards = notebook18.all_card_data()

    official_card_data_by_id = {
        int(card.cardId): card
        for card in official_cards
    }

print("Repository size:", len(repository))
print(
    "Official CardData lookup size:",
    len(official_card_data_by_id),
)

assert len(repository) == 1267
assert len(official_card_data_by_id) == 1267

print("\nPolicy dependencies loaded.")

Repository size: 1267
Official CardData lookup size: 1267

Policy dependencies loaded.


# Cell 9B — Instantiate the policy

In [14]:
policy = BattlePolicy(
    repository=repository,
    official_lookup=official_card_data_by_id,
    debug=True,
)

print("BattlePolicy created.")
print("Debug mode:", policy.debug)

BattlePolicy created.
Debug mode: True


## Cell 10 

In [15]:
assert callable(policy.choose_action)
assert len(policy.repository) == 1267
assert len(policy.official_lookup) == 1267

print("BattlePolicy created successfully.")

BattlePolicy created successfully.


# Cell 11 — Add policy result model

In [16]:
@dataclass(frozen=True)
class PolicyDecision:
    """
    Complete policy result for debugging and evaluation.
    """

    option_index: int
    used_fallback: bool
    reason: str
    ranked_actions: tuple[Any, ...]

# Cell 12 — Upgrade BattlePolicy with safe fallback

In [17]:
@dataclass(slots=True)
class BattlePolicy:
    repository: Any
    official_lookup: dict[int, Any]
    debug: bool = False
    fallback_index: int = 0

    def decide(self, observation: Any) -> PolicyDecision:
        """
        Convert an official Observation into a safe policy decision.
        """

        try:
            snapshot = adapt_observation(
                observation,
                repository=self.repository,
                official_lookup=self.official_lookup,
            )
        except Exception as exc:
            return PolicyDecision(
                option_index=int(self.fallback_index),
                used_fallback=True,
                reason=(
                    "Observation adaptation failed: "
                    f"{type(exc).__name__}: {exc}"
                ),
                ranked_actions=(),
            )

        if snapshot is None:
            return PolicyDecision(
                option_index=int(self.fallback_index),
                used_fallback=True,
                reason="Observation contains no current game state.",
                ranked_actions=(),
            )

        try:
            ranked = rank_legal_actions(
                snapshot,
                repository=self.repository,
            )
        except Exception as exc:
            return PolicyDecision(
                option_index=int(self.fallback_index),
                used_fallback=True,
                reason=(
                    "Action ranking failed: "
                    f"{type(exc).__name__}: {exc}"
                ),
                ranked_actions=(),
            )

        if not ranked:
            return PolicyDecision(
                option_index=int(self.fallback_index),
                used_fallback=True,
                reason="No legal ranked actions were available.",
                ranked_actions=(),
            )

        decision = PolicyDecision(
            option_index=int(ranked[0].option_index),
            used_fallback=False,
            reason=(
                "Selected highest-scoring legal action: "
                f"{ranked[0].semantic_label}"
            ),
            ranked_actions=tuple(ranked),
        )

        if self.debug:
            self.print_decision(decision)

        return decision

    def choose_action(self, observation: Any) -> int:
        """
        Return only the official option index expected by Kaggle.
        """

        return self.decide(observation).option_index

    def print_decision(
        self,
        decision: PolicyDecision,
    ) -> None:
        print("=" * 72)
        print("BattlePolicy Decision")
        print("=" * 72)
        print("Chosen option:", decision.option_index)
        print("Fallback used:", decision.used_fallback)
        print("Reason:", decision.reason)

        if not decision.ranked_actions:
            print("Ranked actions: none")
            return

        print("\nRanked actions:")

        for rank, action in enumerate(
            decision.ranked_actions,
            start=1,
        ):
            print(
                f"{rank:>2}. "
                f"index={action.option_index:<3} "
                f"type={action.option_type:<10} "
                f"score={action.score:>9.3f} "
                f"{action.semantic_label}"
            )

# Cell 13 — Recreate and validate the upgraded policy

In [18]:
policy = BattlePolicy(
    repository=repository,
    official_lookup=official_card_data_by_id,
    debug=True,
    fallback_index=0,
)

assert callable(policy.decide)
assert callable(policy.choose_action)
assert policy.fallback_index == 0

print("Safe BattlePolicy created successfully.")

Safe BattlePolicy created successfully.


# Cell 14 — Test fallback behavior

In [19]:
fallback_decision = policy.decide(None)

print("Option index:", fallback_decision.option_index)
print("Fallback used:", fallback_decision.used_fallback)
print("Reason:", fallback_decision.reason)

assert fallback_decision.option_index == 0
assert fallback_decision.used_fallback is True

print("\nFallback behavior passed.")

Option index: 0
Fallback used: True
Reason: Observation adaptation failed: AttributeError: 'NoneType' object has no attribute 'current'

Fallback behavior passed.


# Cell 15 — Reuse the synthetic snapshot

In [20]:
sample_snapshot = notebook19.sample_snapshot

print("Synthetic snapshot loaded.")
print("Turn:", sample_snapshot.turn)
print(
    "Legal options:",
    len(sample_snapshot.selection.options),
)

assert sample_snapshot.selection is not None
assert len(sample_snapshot.selection.options) == 2

print("\nSynthetic snapshot validation passed.")

Synthetic snapshot loaded.
Turn: 3
Legal options: 2

Synthetic snapshot validation passed.


# Cell 16 — Add direct snapshot decision support

In [21]:
@dataclass(slots=True)
class BattlePolicy:
    repository: Any
    official_lookup: dict[int, Any]
    debug: bool = False
    fallback_index: int = 0

    def decide_snapshot(
        self,
        snapshot: Any,
    ) -> PolicyDecision:
        """
        Rank legal actions from an already adapted BattleSnapshot.
        """

        if snapshot is None:
            return PolicyDecision(
                option_index=int(self.fallback_index),
                used_fallback=True,
                reason="BattleSnapshot was None.",
                ranked_actions=(),
            )

        try:
            ranked = rank_legal_actions(
                snapshot,
                repository=self.repository,
            )
        except Exception as exc:
            return PolicyDecision(
                option_index=int(self.fallback_index),
                used_fallback=True,
                reason=(
                    "Action ranking failed: "
                    f"{type(exc).__name__}: {exc}"
                ),
                ranked_actions=(),
            )

        if not ranked:
            return PolicyDecision(
                option_index=int(self.fallback_index),
                used_fallback=True,
                reason="No legal ranked actions were available.",
                ranked_actions=(),
            )

        decision = PolicyDecision(
            option_index=int(ranked[0].option_index),
            used_fallback=False,
            reason=(
                "Selected highest-scoring legal action: "
                f"{ranked[0].semantic_label}"
            ),
            ranked_actions=tuple(ranked),
        )

        if self.debug:
            self.print_decision(decision)

        return decision

    def decide(self, observation: Any) -> PolicyDecision:
        """
        Convert an official Observation into a safe policy decision.
        """

        try:
            snapshot = adapt_observation(
                observation,
                repository=self.repository,
                official_lookup=self.official_lookup,
            )
        except Exception as exc:
            return PolicyDecision(
                option_index=int(self.fallback_index),
                used_fallback=True,
                reason=(
                    "Observation adaptation failed: "
                    f"{type(exc).__name__}: {exc}"
                ),
                ranked_actions=(),
            )

        return self.decide_snapshot(snapshot)

    def choose_action(self, observation: Any) -> int:
        return self.decide(observation).option_index

    def choose_snapshot_action(
        self,
        snapshot: Any,
    ) -> int:
        return self.decide_snapshot(snapshot).option_index

    def print_decision(
        self,
        decision: PolicyDecision,
    ) -> None:
        print("=" * 72)
        print("BattlePolicy Decision")
        print("=" * 72)
        print("Chosen option:", decision.option_index)
        print("Fallback used:", decision.used_fallback)
        print("Reason:", decision.reason)

        if not decision.ranked_actions:
            print("Ranked actions: none")
            return

        print("\nRanked actions:")

        for rank, action in enumerate(
            decision.ranked_actions,
            start=1,
        ):
            print(
                f"{rank:>2}. "
                f"index={action.option_index:<3} "
                f"type={action.option_type:<10} "
                f"score={action.score:>9.3f} "
                f"{action.semantic_label}"
            )

# Cell 17 — Test successful policy decision

In [22]:
policy = BattlePolicy(
    repository=repository,
    official_lookup=official_card_data_by_id,
    debug=True,
    fallback_index=0,
)

synthetic_decision = policy.decide_snapshot(
    sample_snapshot
)

assert synthetic_decision.used_fallback is False
assert synthetic_decision.option_index == 0
assert synthetic_decision.ranked_actions[0].option_type == "ATTACK"

print("\nSynthetic policy decision passed.")

BattlePolicy Decision
Chosen option: 0
Fallback used: False
Reason: Selected highest-scoring legal action: Attack with Mega Lucario ex

Ranked actions:
 1. index=0   type=ATTACK     score=  259.913 Attack with Mega Lucario ex
 2. index=1   type=END        score=  -25.000 End Turn

Synthetic policy decision passed.
